<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Supervised_Learning_lektioner/Lab_3_Facit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📋 Supervised Learning – Facit (Alla lösningar)

> **OBS: Denna notebook innehåller ALLA svar och lösningar. För elever!**

---

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/  
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se

## Importera allt

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, accuracy_score, 
                               precision_score, recall_score, f1_score,
                               classification_report, roc_curve, auc)
from sklearn.preprocessing import StandardScaler

# Ladda och förbered Iris
iris = load_iris()
X, y = iris.data, iris.target
feature_names = ['sepal_längd', 'sepal_bredd', 'petal_längd', 'petal_bredd']
class_names = ['Setosa', 'Versicolor', 'Virginica']
colors = ['#FF9AA2', '#FFB347', '#87CEEB']

iris_df = pd.DataFrame(iris.data, columns=[
    'sepal_längd_cm', 'sepal_bredd_cm', 
    'petal_längd_cm', 'petal_bredd_cm'
])
iris_df['art'] = [iris.target_names[t] for t in iris.target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Setup klar!")

---
# 📖 LEKTION 1 – Facit

### Quiz 1.1 – Svar:
1. Netflix rekommendationer → **Unsupervised Learning** (eller Supervised om vi har explicita likes)
2. Robot lär sig gå → **Reinforcement Learning**
3. Märkta röntgenbilder → **Supervised Learning** ✓

### Quiz 1.2 – Svar:
1. Supervised learning ger modellen **märkta** exempel
2. Modellen lär sig **mönster** från data
3. Modellen kan förutsäga **svar för** ny data

### Quiz 1.3 – Svar:
1. "Förutsäga huspriset i kronor" → **NEJ**, det är REGRESSION (förutsäga ett tal, inte en kategori)
2. Klasserna i Iris: **Setosa, Versicolor, Virginica**
3. Antal features: **4** (sepal l, sepal w, petal l, petal w)

### Övning 4.1 – Features vs Labels:
1. Features: **sepal_längd, sepal_bredd, petal_längd, petal_bredd** (alla mätningar)
2. Label: **art** (vilken blomma det är)
3. Antal features: **4**

---
# 🌸 LEKTION 2 – Facit

### Quiz 2.1 – Svar (ungefärliga värden):

In [ ]:
# ── Lektion 2 – Beräknade facit ────────────────────────────
print("📊 Statistik per art:")
print()
for art in ['setosa', 'versicolor', 'virginica']:
    data = iris_df[iris_df['art'] == art]
    print(f"🌸 {art.upper()}:")
    for feat in ['sepal_längd_cm', 'sepal_bredd_cm', 'petal_längd_cm', 'petal_bredd_cm']:
        print(f"   {feat}: min={data[feat].min():.2f}, max={data[feat].max():.2f}, medel={data[feat].mean():.2f}")
    print()

print("�� Bästa feature för klassificering: petal_längd_cm")
print("   Setosa: 1.0-1.9 cm (mycket liten)")
print("   Versicolor: 3.0-5.1 cm (mellanstor)")
print("   Virginica: 4.5-6.9 cm (stor)")

---
# 🤖 LEKTION 3 – Facit

### Quiz 3.1 – Beslutsträd:
Blomma med petal_längd=4.5, petal_bredd=1.2:
1. petal_längd < 2.5? → **NEJ** (4.5 > 2.5)
2. Vi tar höger väg
3. petal_bredd < 1.75? → **JA** (1.2 < 1.75)
4. Förutsägelse: **VERSICOLOR**

### Övning 2.1 – Train/Test Split:
1. 20% testdata → 150 × 0.8 = **120 träningsblommor**
2. 30% testdata → 150 × 0.3 = **45 testblommor**
3. 50% testdata → Bara 75 träningsblommor är för lite för en bra modell

### Quiz 3.6:
500 datapunkter × 0.2 = **100 datapunkter** i testset

### Svar på quiz-frågorna:
- **Hyperparameter** = inställning du väljer INNAN träning (t.ex. max_depth)
- **Parameter** = modellens interna värden som lärts upp från data (t.ex. tröskelvärdena i beslutsträdet)
- **Overfitting** = modellen memorerar träningsdata men generaliserare dåligt
- **random_state=42** = Fixerat slumptal för att alltid få samma split (reproducerbarhet)

In [ ]:
# ── Lektion 3 – Accuracy vs Depth facit ────────────────────
depths = range(1, 11)
train_accs = []
test_accs = []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

print("📊 Accuracy per max_depth:")
print(f"{'depth':>6} | {'Träning':>10} | {'Test':>10}")
print("-" * 30)
best_test = max(test_accs)
for d, tr, te in zip(depths, train_accs, test_accs):
    marker = " ← BÄST" if te == best_test else ""
    print(f"{d:>6} | {tr:>9.1%} | {te:>9.1%}{marker}")

print()
print(f"Optimal max_depth för test: {test_accs.index(best_test)+1}")

---
# 🔍 LEKTION 4 – Facit

### Quiz 4.1 – Simulera medicinsk diagnos:
- TP=15, FP=10, FN=5, TN=70
1. **Recall** = 15 / (15+5) = **75%**
2. **Precision** = 15 / (15+10) = **60%**
3. 75% recall = Vi missar 25% av sjuka patienter → Kan vara otillräckligt för allvarlig sjukdom

### Quiz 4.2 – Förklaring av terms:
- **True Positive (TP):** Virginica-blomma korrekt klassificerad som Virginica ✓
- **False Positive (FP):** Versicolor-blomma felaktigt klassificerad som Virginica ✗
- **False Negative (FN):** Virginica-blomma felaktigt klassificerad som Versicolor ✗
- **True Negative (TN):** Versicolor-blomma korrekt klassificerad som Versicolor ✓

### Övning 4.1 – Iris-analys:
- Svåraste att klassificera: **Virginica** (kan blandas med Versicolor)
- Arterna som blandas ihop: **Versicolor och Virginica**
- Setosa är lättast: Den har mycket korta/smala petals

In [ ]:
# ── Lektion 4 – Fullständig Iris-analys ────────────────────
print("📊 Klassificeringsrapport för Iris (max_depth=3):")
print()
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=class_names))

cm = confusion_matrix(y_test, pred)
print("Confusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, annot_kws={'size': 14, 'weight': 'bold'})
ax.set_xlabel('Förutsagd', fontsize=11)
ax.set_ylabel('Faktisk', fontsize=11)
ax.set_title('Confusion Matrix – Iris (max_depth=3)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 💳 LEKTION 5 – Facit

### Quiz 5.1 – Klassoimbalans:
- En modell som alltid säger "Legitimt":
  1. **Accuracy ≈ 99.8%** (nästan alla är legitima)
  2. **Recall för bedrägerier = 0%** (hittar INGET bedrägeri)
  3. **INTE en användbar modell** – värdelös för fraud detection!

### Svar på quiz-frågorna:
- **Klassoimbalans** = Stor skillnad i antal samples per klass (0.17% vs 99.83%)
- **Problem med obalans** = Modellen optimerar accuracy → ignorerar ovanliga klassen
- **Threshold sänks (0.7→0.2)** → Recall ökar, Precision minskar
- **class_weight='balanced'** = Modellen viktar ovanliga klasser (bedrägeri) högre vid träning
- **False Negative** = Missat bedrägeri → Kunden drabbas av stöld utan skydd
- **False Positive** = Falskt alarm → Kundens kort blockeras i onödan (irriterande men säkrare)
- **Precision = 50/1000 = 5%** (om 50 av 1000 flaggade är verkliga bedrägerier)

### Threshold-rekommendation:
- För en bank: **Låg threshold (0.2-0.4)** är ofta lämplig
- Bättre att ha falskt alarm (FP) än att missa bedrägeri (FN)
- Kunden kan bekräfta legitimt köp, men återfår aldrig stulet pengar lika enkelt

In [ ]:
# ── Lektion 5 – Sammanfattning av alla mått ─────────────────
print("🏆 KURSSUMMERING – Viktiga begrepp:")
print()
print("=" * 60)
print("LEKTION 1 – Grundbegrepp")
print("-" * 60)
print("  Supervised Learning: Lär sig från märkta exempel")
print("  Features: Mätningar/egenskaper (input)")
print("  Labels:   Rätt svar (output)")
print("  Train/Test: Dela data för ärlig utvärdering")
print()
print("LEKTION 2 – Dataanalys")
print("-" * 60)
print("  pandas.describe(): Statistisk sammanfattning")
print("  Histogram: Fördelning av en feature")
print("  Scatterplot: Relation mellan två features")
print("  Klassbalans: Lika många per klass = bra!")
print()
print("LEKTION 3 – Träna modell")
print("-" * 60)
print("  DecisionTreeClassifier: Beslutsträd-modell")
print("  max_depth: Hyperparameter (kontrollerar komplexitet)")
print("  Accuracy: Andel korrekta förutsägelser")
print("  Overfitting: Bra på träning, dålig på test")
print()
print("LEKTION 4 – Evaluering")
print("-" * 60)
print("  Confusion Matrix: Visar exakt typ av fel")
print("  Precision: Av flaggade, hur många rätt?")
print("  Recall:    Av alla rätta, hur många hittade vi?")
print("  F1-score:  Balanserat mått")
print()
print("LEKTION 5 – Verkligt problem")
print("-" * 60)
print("  Klassoimbalans: Olik fördelning av klasser")
print("  Threshold: Gräns för klassificering (0-1)")
print("  ROC-kurva: Visualiserar Precision/Recall trade-off")
print("  Random Forest: Ensemble av beslutsträd")
print("=" * 60)